# 3.4 Survival Analysis

**Class 5.** Among people **with diabetes**, how long does it take to develop **chronic kidney disease (CKD)**, and who develops it sooner? Questions about **how long until an event** need **survival analysis** (also called *time-to-event* analysis). The name comes from studies of time until death, but the event can be anything: here, a CKD diagnosis.

### By the end of this notebook you'll be able to

1. Build a **time-to-event** table: one row per person with a start, an end, and an event indicator
2. Explain **censoring**, and why we can't just drop people who never had the event
3. Estimate and plot **Kaplan–Meier** curves and compare groups with the **log-rank test**
4. Fit a **Cox regression** and interpret **hazard ratios (HR)**, and check its main assumption

| | |
|---|---|
| ⏱️ **Time** | 60–75 minutes |
| 📂 **Uses** | `condition_arranged.csv`, `visit_arranged.csv` (1.2), `person_study.csv` (1.1), `measurements_wide.csv` (2.0) |
| 💾 **Saves** | nothing |

**Before you start:**

- Run **1.2 survival_data** and **2.0 clean_and_join**

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Key concepts

### Time zero, the event, and censoring

For each person we need:

- **Time zero:** the first **diabetes** diagnosis
- **The event:** the first **CKD** diagnosis, if any (`status = 1`)
- **The end of follow-up:** the CKD date, or, if they never developed CKD, their **last visit** (`status = 0`)

People with `status = 0` are **censored**: we know they were CKD-free **at least** until their last visit, but not what happened after. If we simply dropped them, or treated them as never getting CKD, we'd get the wrong answer. Survival methods use the time each person **was** observed.

### Kaplan–Meier curves

A **Kaplan–Meier (KM) curve** shows the share of people still **event-free** over time. It starts at 1 (100%) and steps down each time someone has the event. Censored people leave the calculation without causing a step. Curves for different groups can be compared with the **log-rank test**, whose p-value asks whether the curves differ more than chance would explain.

### Hazards and the Cox model

The **hazard** is the instantaneous risk of the event at a given time, among people who haven't had it yet. A **Cox proportional-hazards model** estimates **hazard ratios (HR)**, which you read much like odds ratios:

| HR | Means |
|---|---|
| **> 1** | higher risk, CKD sooner (e.g. HR 1.5 = 50% higher hazard) |
| **= 1** | no association |
| **< 1** | lower risk |

It assumes each HR stays **constant over time** (the *proportional hazards* assumption), which we check at the end.

### Predictors must come before the outcome

Predictors should describe people **at or before time zero**. Using something that is only known **after** time zero, like the stage of CKD someone eventually reaches, would be circular: it's part of the outcome.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

We load two packages for survival analysis (installed automatically by `aou_setup.R` if missing):

- **survival:** the core tools, `Surv()`, `survfit()`, `coxph()`
- **survminer:** nicer survival plots with `ggsurvplot()`

In [ ]:
library(survival)
library(survminer)   # ggsurvplot()

condition_arranged <- load_data("condition_arranged.csv")
visit_arranged     <- load_data("visit_arranged.csv")
person_study       <- load_data("person_study.csv")
measurements_wide  <- load_data("measurements_wide.csv")

## Prepare: label the diagnoses and find each person's last visit

Each row of `condition_arranged` is one diagnosis. We **flag** it from the condition's name, the same idea as the group flag in 2.0, but done with a rule:

- `is_diabetes`: the name mentions diabetes
- `is_ckd`: the name mentions kidney or renal disease

A diagnosis like *"CKD stage 3 due to type 2 diabetes"* counts as **both**. `str_to_lower()` makes the name lower-case so capitals don't matter.

From `visit_arranged` we keep each person's **last** visit date: `group_by(person_id)` then `summarise(max(…))`. `rename()` changes `PERSON_ID` to `person_id` to match the other tables.

In [ ]:
survival_conditions <- condition_arranged %>%
  transmute(
    person_id,
    condition_start_date = as.Date(condition_start_datetime),
    is_diabetes = str_detect(str_to_lower(standard_concept_name), "diabet"),
    is_ckd      = str_detect(str_to_lower(standard_concept_name), "kidney|renal")
  )

last_visits <- visit_arranged %>%
  rename(person_id = PERSON_ID) %>%
  group_by(person_id) %>%
  summarise(last_visit_date = max(as.Date(visit_end_datetime), na.rm = TRUE))

survival_conditions %>% count(is_diabetes, is_ckd)

**🔍 Read the output:** most diagnoses are diabetes only; a smaller number are CKD, or both.

## Step 1: First diabetes and first CKD date per person

For each person (`group_by(person_id)`), `summarise()` finds the **earliest** date among their diabetes diagnoses, and among their CKD diagnoses. `condition_start_date[is_diabetes]` means "the dates where `is_diabetes` is TRUE".

When someone has **no** CKD diagnoses, `min()` of nothing gives `Inf` (infinity); the `mutate(across(…))` line turns those into `NA` (missing), which is what they really are. `across()` applies the same change to several columns at once.

In [ ]:
first_dates <- survival_conditions %>%
  group_by(person_id) %>%
  summarise(
    diagnosis_date = suppressWarnings(min(condition_start_date[is_diabetes])),
    ckd_date       = suppressWarnings(min(condition_start_date[is_ckd]))
  ) %>%
  mutate(across(c(diagnosis_date, ckd_date), ~ if_else(is.infinite(.x), as.Date(NA), .x))) %>%
  filter(!is.na(diagnosis_date))

head(first_dates)

## Step 2: Time to event and censoring

Now we build the three survival columns:

- **Drop prevalent cases.** People whose CKD came **before or on** their diabetes diagnosis already had it at time zero, so they weren't *at risk* of developing it. We remove them.
- `status`: **1** if CKD happened, **0** if censored
- `end_date`: the CKD date, or the last visit for people without CKD
- `time_years`: the years from diagnosis to the end of follow-up (subtracting dates gives days; ÷ 365.25 gives years)

We drop anyone with missing or zero follow-up time, which usually means no visits after diagnosis.

In [ ]:
surv_data <- first_dates %>%
  left_join(last_visits, by = "person_id") %>%
  filter(is.na(ckd_date) | ckd_date > diagnosis_date) %>%
  mutate(
    status     = if_else(!is.na(ckd_date), 1, 0),
    end_date   = if_else(status == 1, ckd_date, last_visit_date),
    time_years = as.numeric(end_date - diagnosis_date) / 365.25
  ) %>%
  filter(!is.na(time_years), time_years > 0)

surv_data %>% count(status)

**🔍 Read the output:** how many people developed CKD (`status = 1`) and how many were censored (`0`)? Usually most are censored, which is exactly why we need survival methods.

## Step 3: Add baseline characteristics

We add things known **at time zero**:

- **age at diagnosis** and **gender**, from `person_study.csv`
- **BMI** and **HbA1c**, from `measurements_wide.csv`, plus an HbA1c group: **< 7%** (a common treatment target for diabetes) vs **≥ 7%**

> ⚠️ **Simplification:** these are each person's **most recent** BMI and HbA1c, which may have been measured **after** diagnosis. A real study would use the value closest to, and before, `diagnosis_date`. Keep this in mind as a limitation.

In [ ]:
surv_data <- surv_data %>%
  left_join(person_study %>% transmute(person_id, date_of_birth = as.Date(date_of_birth), gender), by = "person_id") %>%
  left_join(measurements_wide %>% select(person_id, BMI, HbA1c), by = "person_id") %>%
  mutate(
    age_at_diagnosis = as.numeric(diagnosis_date - date_of_birth) / 365.25,
    hba1c_group = factor(if_else(HbA1c >= 7, "HbA1c >= 7%", "HbA1c < 7%"))
  )

glimpse(surv_data)

## Step 4: Kaplan–Meier curves

- `Surv(time_years, status)` combines follow-up time and the event indicator into one **survival object**
- `survfit(Surv(…) ~ hba1c_group)` estimates a KM curve for each HbA1c group
- `ggsurvplot()` draws the curves, with 95% confidence bands (`conf.int`), the log-rank p-value (`pval`), and a **risk table** underneath showing how many people are still being followed at each time

`options(repr.plot.width, repr.plot.height)` makes the plot bigger in the notebook.

> ⚠️ The **risk table** shows small numbers at long follow-up times. Counts of 1 to 20 **can't be shared**, so don't screenshot it.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 9)

km_fit <- survfit(Surv(time_years, status) ~ hba1c_group, data = surv_data)

ggsurvplot(
  km_fit, data = surv_data,
  conf.int = TRUE, pval = TRUE, risk.table = TRUE,
  xlim = c(0, 10), break.time.by = 2,
  xlab = "Years since diabetes diagnosis",
  ylab = "Proportion CKD-free",
  ggtheme = theme_bw(base_size = 14)
)

**🔍 Read the output:**

- Each curve starts at 1.0 (everyone CKD-free) and steps down when people develop CKD
- A curve that **drops faster** means that group develops CKD **sooner**
- Read the **share CKD-free at 5 years** for each group from the y-axis
- The **p-value** (log-rank test) tells you whether the curves differ more than chance would explain
- Confidence bands widen at later times, as fewer people are still followed

### ✍️ Your turn

1. Make Kaplan–Meier curves by **gender** instead of HbA1c group.
2. Print `km_fit` to see the **median** time to CKD for each group. What does `NA` mean there?

<details>
<summary><b>Show answer</b></summary>

1.
```r
km_gender <- survfit(Surv(time_years, status) ~ gender, data = surv_data)
ggsurvplot(km_gender, data = surv_data, pval = TRUE, conf.int = TRUE, xlim = c(0, 10))
```

2.
```r
km_fit
```
The **median** is the time by which half the group has developed CKD. `NA` means fewer than half developed CKD during follow-up, so the median can't be estimated, which is common when the event is fairly rare.

</details>

In [ ]:
# Your code here

## Step 5: Cox proportional-hazards model

`coxph()` fits a Cox model with several predictors at once. Each **hazard ratio** is adjusted for the others. `tidy(exponentiate = TRUE)` converts the results to HRs, like ORs in 3.2.

- **Age and BMI:** HR per **1-unit** increase (one year; one kg/m²)
- **Gender and HbA1c group:** HR compared with the **reference** category (Female; HbA1c < 7%)

In [ ]:
cox_model <- coxph(Surv(time_years, status) ~ age_at_diagnosis + gender + BMI + hba1c_group,
                   data = surv_data)

broom::tidy(cox_model, exponentiate = TRUE, conf.int = TRUE) %>%
  select(term, estimate, conf.low, conf.high, p.value)

**🔍 Read the output:** `estimate` is the **hazard ratio**; check whether each 95% CI includes 1.

**Concordance (C-index)** measures how well the model **ranks** people by risk: of all pairs of people, how often does the person who developed CKD sooner have the higher predicted risk? 0.5 is no better than chance; 1 is perfect; above about 0.7 is usually considered good.

In [ ]:
summary(cox_model)$concordance

## Step 6: Check the proportional-hazards assumption

The Cox model assumes each HR is **the same at every point in time**. `cox.zph()` tests this for each variable and overall (`GLOBAL`). A **small p-value** (< 0.05) suggests that variable's effect **changes over time**. With very large samples, small, unimportant changes can be flagged; `plot(cox.zph(cox_model))` shows how big they are.

In [ ]:
cox.zph(cox_model)

### ✍️ Your turn

Write a plain-language sentence for the **HbA1c group** hazard ratio. Then add **age group** instead of age as a continuous variable. (Hint: make it with `cut()`, as in 2.0.)

<details>
<summary><b>Show answer</b></summary>

*"People with diabetes whose HbA1c was 7% or higher had **[HR]** times the hazard of developing CKD compared with those below 7%, after adjusting for age, gender, and BMI (95% CI … to …)."*

```r
surv_data <- surv_data %>%
  mutate(age_group = cut(age_at_diagnosis, breaks = c(0, 35, 45, 100), right = FALSE,
                         labels = c("<35", "35-44", "45+")))

coxph(Surv(time_years, status) ~ age_group + gender + BMI + hba1c_group, data = surv_data) %>%
  broom::tidy(exponentiate = TRUE, conf.int = TRUE)
```
Each age-group HR is now compared with the youngest group.

</details>

In [ ]:
# Your code here

## Discuss

1. Why must predictors be measured **at or before** time zero?
2. What other baseline factors might matter? (Blood pressure, medications, kidney function at diagnosis…)
3. **Competing risks:** some people die before they could develop CKD. How might that affect these results?
4. We used each person's last visit as the end of follow-up. What if someone kept getting care **outside** the hospitals that share data with All of Us?

## ⚠️ Common errors

| Problem | Fix |
|---|---|
| *there is no package called 'survminer'* | Rerun `source("../aou_setup.R")`: it installs missing packages |
| `surv_data` has very few rows | Check the counts after each step; files from 1.2 may be missing |
| *Loglik converged before variable…* | A predictor perfectly separates events; check that variable |

## ✅ Key takeaways

- Survival analysis needs **time zero**, an **event**, and **follow-up time**, with **censoring** for people without the event
- **Kaplan–Meier** curves describe time to event; the **log-rank test** compares groups
- **Cox regression** gives adjusted **hazard ratios**; check the **proportional-hazards** assumption
- Predictors must be known **at time zero**

🎉 **You've finished the series!** You've gone from a database query to a set of analyses you'd see in a clinical research paper. To use these methods on your own question, start again at **1.1** with your own cohort, or build it in **Data Explorer**.